# RTDA Keras Model → hls4ml

Loads the pre-trained RTDA per-track model from `data/*.txt` weights, runs streaming Keras inference, and converts to Vitis HLS via hls4ml.

**Sections**
1. Keras model + weights
2. Reference outputs (save `keras_s2out.txt`)
3. hls4ml conversion & float32 compile
4. C-simulation & comparison
5. C-synthesis *(run when ready)*
6. Synthesis reports


In [1]:
import os, re, types, warnings
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tf_keras                           # Keras 2 — required by hls4ml 0.8.x
from tf_keras import Input, Model
from tf_keras.layers import Dense, Add, LeakyReLU

import hls4ml

print(f'tf_keras {tf_keras.__version__}  hls4ml {hls4ml.__version__}  numpy {np.__version__}')


E0000 00:00:1776800762.440172 3770074 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776800762.444193 3770074 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776800762.456615 3770074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776800762.456632 3770074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776800762.456634 3770074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776800762.456635 3770074 computation_placer.cc:177] computation placer already registered. Please check linka

tf_keras 2.19.0  hls4ml 0.8.1  numpy 1.26.4


In [2]:
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
REPO_ROOT    = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data').is_dir() else NOTEBOOK_DIR.parent
DATA_DIR     = REPO_ROOT / 'data'
HLS_OUT_DIR  = str(REPO_ROOT / 'hls_projects' / 'rtda_fp32_hls4ml')
OUT_DIR      = NOTEBOOK_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

INPUT_SIZE   = 6
HIDDEN_SIZE  = 128
OUTPUT_SIZE  = 27
N_TRACKS     = 50
WARMUP       = 3      # first 3 streaming tracks use zero states → exclude from comparison
LEAKY_ALPHA  = 0.1

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'HLS_OUT   : {HLS_OUT_DIR}')
print(f'outputs/  : {OUT_DIR}')


REPO_ROOT : /home/synthara/VersalPrjs/LDRD/quant_rtda/rtda_demo
HLS_OUT   : /home/synthara/VersalPrjs/LDRD/quant_rtda/rtda_demo/hls_projects/rtda_fp32_hls4ml
outputs/  : /home/synthara/VersalPrjs/LDRD/quant_rtda/rtda_demo/punit/outputs


## 1. Keras Model

Architecture: Embed block (2 dense) + 3 Solver blocks (4 dense each), all followed by LeakyReLU(α=0.1).  Each Solver Dense0 is split into two 128×128 sub-layers (`_curr` + `_prev`) to avoid a 256-wide input that hls4ml cannot handle.

Inputs: `track_in` (6), `state0/1/2` (128 each)  
Outputs: `s2_out` (128), `emb_out` (128), `s0_out` (128), `s1_out` (128)

In [3]:
def build_rtda_per_track():
    def lrelu(x, name=None):
        return LeakyReLU(alpha=LEAKY_ALPHA, name=name)(x)

    def solver_block(curr, state, p):
        """Dense0 is split: W0_left(curr) + W0_right(state) then Add."""
        x = lrelu(Add(name=f'{p}_d0_add')([
                Dense(HIDDEN_SIZE, use_bias=True,  name=f'{p}_d0_curr')(curr),
                Dense(HIDDEN_SIZE, use_bias=False, name=f'{p}_d0_prev')(state),
            ]), name=f'{p}_d0_act')
        for i in range(1, 4):
            x = lrelu(Dense(HIDDEN_SIZE, name=f'{p}_d{i}')(x), name=f'{p}_d{i}_act')
        return x

    ti = Input(shape=(INPUT_SIZE,),  name='track_in')
    s0 = Input(shape=(HIDDEN_SIZE,), name='state0')
    s1 = Input(shape=(HIDDEN_SIZE,), name='state1')
    s2 = Input(shape=(HIDDEN_SIZE,), name='state2')

    e  = lrelu(Dense(HIDDEN_SIZE, name='emb_d0')(ti), name='emb_d0_act')
    eo = lrelu(Dense(HIDDEN_SIZE, name='emb_d1')(e),  name='emb_d1_act')
    o0 = solver_block(eo, s0, 's0')
    o1 = solver_block(o0, s1, 's1')
    o2 = solver_block(o1, s2, 's2')

    return Model(inputs=[ti, s0, s1, s2],
                 outputs=[o2, eo, o0, o1],    # s2_out, emb_out, s0_out, s1_out
                 name='rtda_per_track')

model = build_rtda_per_track()
model.summary()


Model: "rtda_per_track"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 track_in (InputLayer)       [(None, 6)]                  0         []                            
                                                                                                  
 emb_d0 (Dense)              (None, 128)                  896       ['track_in[0][0]']            
                                                                                                  
 emb_d0_act (LeakyReLU)      (None, 128)                  0         ['emb_d0[0][0]']              
                                                                                                  
 emb_d1 (Dense)              (None, 128)                  16512     ['emb_d0_act[0][0]']          
                                                                                     

In [4]:
# ── Load weights from data/*.txt ────────────────────────────────────────────
def load_parts(stem, n_parts, shape):
    return np.concatenate([
        np.loadtxt(DATA_DIR / f'{stem}_part{i}.txt') for i in range(n_parts)
    ]).reshape(shape).astype(np.float32)

def load_bias(stem):
    return np.loadtxt(DATA_DIR / f'{stem}.txt').astype(np.float32)

# Embed
embed_W0 = np.loadtxt(DATA_DIR / 'embed_dense_0_weights.txt').reshape(8, 128)[:6, :].astype(np.float32)
embed_B0 = load_bias('embed_dense_0_bias')
embed_W1 = load_parts('embed_dense_1_weights', 2, (128, 128))
embed_B1 = load_bias('embed_dense_1_bias')

# Solvers
def load_solver(s):
    W0 = load_parts(f'solver_{s}_dense_0_weights', 4, (256, 128))
    B0 = load_bias(f'solver_{s}_dense_0_bias')
    layers = [{'W': W0, 'B': B0}]
    for d in range(1, 4):
        layers.append({'W': load_parts(f'solver_{s}_dense_{d}_weights', 2, (128, 128)),
                        'B': load_bias(f'solver_{s}_dense_{d}_bias')})
    return layers

sw = [load_solver(s) for s in range(3)]

# Output layer
out_W = np.loadtxt(DATA_DIR / 'output_weights.txt').reshape(128, 27).astype(np.float32)
out_B = load_bias('output_bias')

# ── Set weights ─────────────────────────────────────────────────────────────
model.get_layer('emb_d0').set_weights([embed_W0, embed_B0])
model.get_layer('emb_d1').set_weights([embed_W1, embed_B1])

for p, s in [('s0', 0), ('s1', 1), ('s2', 2)]:
    W0 = sw[s][0]['W']                               # (256, 128)
    model.get_layer(f'{p}_d0_curr').set_weights([W0[:128, :], sw[s][0]['B']])
    model.get_layer(f'{p}_d0_prev').set_weights([W0[128:, :]])
    for d in range(1, 4):
        model.get_layer(f'{p}_d{d}').set_weights([sw[s][d]['W'], sw[s][d]['B']])

print('All weights loaded.')


All weights loaded.


## 2. Reference Outputs

Run the stateful streaming loop over all 50 input tracks.  
State initialises at zero; the first `WARMUP=3` tracks are excluded from comparisons.  
Results are saved to `outputs/keras_s2out.txt` (50 × 128) and `outputs/keras_27dim.txt` (27,) for later cross-check.

In [5]:
# Load input tracks (embed_input.txt pads 6→8 for AIE; drop the padding)
raw    = np.loadtxt(DATA_DIR / 'embed_input.txt', dtype=np.float32)
tracks = raw[:N_TRACKS * 8].reshape(N_TRACKS, 8)[:, :INPUT_SIZE]   # (50, 6)

# Streaming inference
st = [np.zeros((1, HIDDEN_SIZE), np.float32) for _ in range(3)]
keras_s2out  = np.zeros((N_TRACKS, HIDDEN_SIZE), np.float32)
keras_states = []   # per-track input states saved for hls4ml re-run

for j in range(N_TRACKS):
    keras_states.append(tuple(s.copy() for s in st))
    s2, eo, o0, o1 = [o.numpy().reshape(HIDDEN_SIZE)
                      for o in model([tracks[j:j+1], *st], training=False)]
    keras_s2out[j] = s2
    st[0], st[1], st[2] = eo.reshape(1,-1), o0.reshape(1,-1), o1.reshape(1,-1)

# Final 27-dim output: average s2 over post-warmup tracks → linear output layer
keras_27dim = keras_s2out[WARMUP:].mean(axis=0) @ out_W + out_B

# Save
np.savetxt(OUT_DIR / 'keras_s2out.txt',  keras_s2out, fmt='%.8e',
           header=f'Keras s2_out  shape ({N_TRACKS}, {HIDDEN_SIZE})')
np.savetxt(OUT_DIR / 'keras_27dim.txt',  keras_27dim, fmt='%.8e',
           header=f'Keras final 27-dim output')

print(f'keras_s2out : {keras_s2out.shape}  → outputs/keras_s2out.txt')
print(f'keras_27dim : {keras_27dim.shape} → outputs/keras_27dim.txt')

# Quick sanity: compare post-warmup against AIE hw reference
aie_s2 = np.loadtxt(DATA_DIR / 'aieml10_output_aie.txt', dtype=np.float32).reshape(N_TRACKS, HIDDEN_SIZE)
diffs  = [np.max(np.abs(keras_s2out[j] - aie_s2[j])) for j in range(WARMUP, N_TRACKS)]
print(f'Keras vs AIE hw (tracks {WARMUP}-{N_TRACKS-1}): max|diff| = {max(diffs):.2e}')


keras_s2out : (50, 128)  → outputs/keras_s2out.txt
keras_27dim : (27,) → outputs/keras_27dim.txt
Keras vs AIE hw (tracks 3-49): max|diff| = 1.25e-06


## 3. hls4ml Conversion

### Key config flags
| Flag | Value | Notes |
|---|---|---|
| `IO_TYPE` | `'io_stream'` | HLS streams between layers — lower BRAM/LUT, pipelined; use `'io_parallel'` for lower latency at higher area cost |
| `STRATEGY` | `'Resource'` | Multipliers shared across time steps; `'Latency'` unrolls fully |
| `REUSE_FACTOR` | `64` | Each multiplier reused 64× → ~64× fewer DSPs vs RF=1; increase for tighter area, decrease for throughput |

### Float32 note
hls4ml 0.8.1 ignores `config['Model']['Precision'] = 'float'` — `defines.h` always gets `ap_fixed<16,6>`. The `write()` method is monkey-patched to replace every `ap_fixed<…>` with `float` in `firmware/defines.h` after every write cycle.

In [6]:
# ── hls4ml config ────────────────────────────────────────────────────────────
IO_TYPE      = 'io_stream'   # 'io_stream' | 'io_parallel'
STRATEGY     = 'Resource'    # 'Resource'  | 'Latency'
REUSE_FACTOR = 64            # 1 = fully unrolled; 128 = maximally shared for 128-wide layers

config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')
config['Model']['Strategy']    = STRATEGY
config['Model']['ReuseFactor'] = REUSE_FACTOR
config['Model']['Precision']   = 'ap_fixed<32,8>'   # placeholder — patched to float below

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config   = config,
    output_dir   = HLS_OUT_DIR,
    backend      = 'Vitis',
    part         = 'xcve2802-vsvh1760-2MP-e-S',
    io_type      = IO_TYPE,
)

# ── Patch helper (also used in Section 5) ───────────────────────────────────
def apply_firmware_patch(fw_dir):
    """Expose layer7/23/39 as output ports. Safe to re-run."""
    PATCHED_H_SIG = 'hls::stream<layer39_t> &layer39_out'
    h_path   = fw_dir / 'myproject.h'
    cpp_path = fw_dir / 'myproject.cpp'
    if not h_path.exists(): return
    if PATCHED_H_SIG in h_path.read_text(): return  # already patched
    h_path.write_text('''
#ifndef MYPROJECT_H_
#define MYPROJECT_H_

#include "ap_fixed.h"
#include "ap_int.h"
#include "hls_stream.h"

#include "defines.h"

// layer7_out  = emb_out  -> feed back as state0 next track
// layer23_out = s0_out   -> feed back as state1 next track
// layer39_out = s1_out   -> feed back as state2 next track
void myproject(
    hls::stream<input_t>   &track_in,
    hls::stream<input8_t>  &state0,
    hls::stream<input24_t> &state1,
    hls::stream<input40_t> &state2,
    hls::stream<result_t>  &layer55_out,
    hls::stream<layer7_t>  &layer7_out,
    hls::stream<layer23_t> &layer23_out,
    hls::stream<layer39_t> &layer39_out
);

#endif
''')
    txt = cpp_path.read_text()
    txt = re.sub(
        r'void myproject\([^)]+\)',
        ('void myproject(\n'
         '    hls::stream<input_t>   &track_in,\n'
         '    hls::stream<input8_t>  &state0,\n'
         '    hls::stream<input24_t> &state1,\n'
         '    hls::stream<input40_t> &state2,\n'
         '    hls::stream<result_t>  &layer55_out,\n'
         '    hls::stream<layer7_t>  &layer7_out,\n'
         '    hls::stream<layer23_t> &layer23_out,\n'
         '    hls::stream<layer39_t> &layer39_out\n)'),
        txt, flags=re.DOTALL)
    txt = txt.replace(
        'axis port=track_in,state0,state1,state2,layer55_out',
        'axis port=track_in,state0,state1,state2,layer55_out,layer7_out,layer23_out,layer39_out')
    for decl in [
        'hls::stream<layer7_t> layer7_out("layer7_out");\n    #pragma HLS STREAM variable=layer7_out depth=1\n',
        'hls::stream<layer23_t> layer23_out("layer23_out");\n    #pragma HLS STREAM variable=layer23_out depth=1\n',
        'hls::stream<layer39_t> layer39_out("layer39_out");\n    #pragma HLS STREAM variable=layer39_out depth=1\n',
    ]:
        txt = txt.replace(decl, '')
    cpp_path.write_text(txt)

    # Fix myproject_bridge.cpp — the Python bridge that hls_model.compile() uses.
    # It calls myproject() with 5 args; patch both call sites to pass 3 extra dummy
    # output streams for layer7_out, layer23_out, layer39_out.
    bridge = fw_dir.parent / 'myproject_bridge.cpp'
    if bridge.exists() and 'layer7_out_ap' not in bridge.read_text():
        OLD_CALL = 'myproject(track_in_ap,state0_ap,state1_ap,state2_ap,layer55_out_ap);'
        NEW_CALL = (
            'hls::stream<layer7_t>  layer7_out_ap;\n'
            '    hls::stream<layer23_t> layer23_out_ap;\n'
            '    hls::stream<layer39_t> layer39_out_ap;\n'
            '    myproject(track_in_ap,state0_ap,state1_ap,state2_ap,layer55_out_ap,\n'
            '              layer7_out_ap,layer23_out_ap,layer39_out_ap);'
        )
        bridge.write_text(bridge.read_text().replace(OLD_CALL, NEW_CALL))

    # Fix myproject_test.cpp (Vitis HLS csim testbench — same 5-arg call)
    test_path = fw_dir.parent / 'myproject_test.cpp'
    if test_path.exists() and 'layer7_out_dummy' not in test_path.read_text():
        ttxt = test_path.read_text().replace(
            'myproject(track_in,state0,state1,state2,layer55_out);',
            'hls::stream<layer7_t>  layer7_out_dummy;\n'
            '    hls::stream<layer23_t> layer23_out_dummy;\n'
            '    hls::stream<layer39_t> layer39_out_dummy;\n'
            '    myproject(track_in,state0,state1,state2,layer55_out,\n'
            '              layer7_out_dummy,layer23_out_dummy,layer39_out_dummy);')
        test_path.write_text(ttxt)

# ── Monkey-patch write() — fires on every compile() / build() call ──────────
# Applies three fixes hls4ml 0.8.1 gets wrong for Vitis 2024.2:
#   1. defines.h       : ap_fixed -> float
#   2. myproject.h/.cpp: expose layer7/23/39 as output ports
#   3. build_prj.tcl   : remove -maximum_size (invalid option in Vitis 2024.2)
_orig_write = hls_model.write.__func__

def _patched_write(self):
    _orig_write(self)
    fw = Path(HLS_OUT_DIR) / 'firmware'
    defs = fw / 'defines.h'
    if defs.exists():
        defs.write_text(re.sub(r'ap_fixed<[^>]+>', 'float', defs.read_text()))
    apply_firmware_patch(fw)
    tcl = Path(HLS_OUT_DIR) / 'build_prj.tcl'
    if tcl.exists() and '-maximum_size' in tcl.read_text():
        tcl.write_text(tcl.read_text().replace(
            'catch {config_array_partition -maximum_size 4096}',
            '# config_array_partition -maximum_size removed: not valid in Vitis HLS 2024.2'))

hls_model.write = types.MethodType(_patched_write, hls_model)
_patched_write(hls_model)  # apply to already-written files now

print(f'io_type={IO_TYPE}  strategy={STRATEGY}  reuse_factor={REUSE_FACTOR}')
print('Patches on write(): [1] float precision  [2] output ports  [3] TCL fix')


Interpreting Model
Topology:
Layer name: track_in, layer type: InputLayer, input shapes: [[None, 6]], output shape: [None, 6]
Layer name: emb_d0, layer type: Dense, input shapes: [[None, 6]], output shape: [None, 128]
Layer name: emb_d0_act, layer type: LeakyReLU, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: emb_d1, layer type: Dense, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: emb_d1_act, layer type: LeakyReLU, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: state0, layer type: InputLayer, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: s0_d0_curr, layer type: Dense, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: s0_d0_prev, layer type: Dense, input shapes: [[None, 128]], output shape: [None, 128]
Layer name: s0_d0_add, layer type: Merge, input shapes: [[None, 128], [None, 128]], output shape: [None, 128]
Layer name: s0_d0_act, layer type: LeakyReLU, input shapes: [[None, 128]], outp

In [7]:
%%time
# Compiles generated C++ into a .so for Python-level csim (uses g++, not vitis_hls).
# Applies all patches via write() before compiling.
try:
    hls_model.compile()
    HLS_COMPILED = True
    line = next((l.strip() for l in open(HLS_OUT_DIR + '/firmware/defines.h')
                 if 'input_t' in l and 'typedef' in l), '?')
    print(f'Compiled OK.  defines.h input_t: {line}')
except Exception as e:
    print(f'Compile failed: {e}')
    HLS_COMPILED = False


Writing HLS project
Done
Compiled OK.  defines.h input_t: typedef nnet::array<float, 6*1> input_t;
CPU times: user 4.51 s, sys: 75 ms, total: 4.59 s
Wall time: 6.75 s


## 4. C-Simulation & Comparison

hls4ml 0.8.1 `predict()` returns only the **first model output** (`s2_out`, 128-dim) for multi-output models — this is a known limitation.  The comparison is therefore on `s2_out` across all 50 tracks using the same per-track states that were computed by Keras.

In [8]:
if HLS_COMPILED:
    hls_s2out = np.zeros((N_TRACKS, HIDDEN_SIZE), np.float32)

    for j in range(N_TRACKS):
        st0, st1, st2 = keras_states[j]
        hls_s2out[j] = hls_model.predict([
            np.ascontiguousarray(tracks[j:j+1]),
            np.ascontiguousarray(st0),
            np.ascontiguousarray(st1),
            np.ascontiguousarray(st2),
        ]).reshape(HIDDEN_SIZE)

    # Save hls outputs
    np.savetxt(OUT_DIR / 'hls_s2out.txt', hls_s2out, fmt='%.8e',
               header=f'hls4ml csim s2_out  shape ({N_TRACKS}, {HIDDEN_SIZE})')

    # Compare
    diff = np.abs(hls_s2out - keras_s2out)            # (50, 128)
    diff_post = diff[WARMUP:]                          # exclude warmup tracks

    print(f'hls4ml csim vs keras_s2out.txt')
    print(f'  max |diff| all tracks    : {diff.max():.2e}')
    print(f'  max |diff| post-warmup   : {diff_post.max():.2e}')
    print(f'  RMSE        post-warmup  : {np.sqrt((diff_post**2).mean()):.2e}')

    tol = 1e-4
    status = 'PASS ✓' if diff_post.max() < tol else f'FAIL — exceeds {tol}'
    print(f'  [{status}]')
    print(f'Saved outputs/hls_s2out.txt')
else:
    print('Skipped — compile failed or Vitis not sourced.')


hls4ml csim vs keras_s2out.txt
  max |diff| all tracks    : 5.22e-07
  max |diff| post-warmup   : 5.22e-07
  RMSE        post-warmup  : 5.42e-08
  [PASS ✓]
Saved outputs/hls_s2out.txt


## 5. Firmware Patch & `rtda_top.cpp`

### Why this is needed
`myproject()` as generated by hls4ml exposes only **one output** (`layer55_out` = `s2_out`).
The three intermediate activations that must feed back as states for the next track
(`layer7_out` = emb_out, `layer23_out` = s0_out, `layer39_out` = s1_out)
are internal streams that get consumed and disappear.

The patch below adds them as proper output ports.  It is also wired into the `write()` monkey-patch so every `hls_model.compile()` re-applies it.

### Mapping of the 3 extra outputs
| Port | Layer | Feeds back as |
|---|---|---|
| `layer7_out`  | `emb_d1_act` — embed block final output   | `state0` for track j+1 |
| `layer23_out` | `s0_d3_act`  — solver0 final output       | `state1` for track j+1 |
| `layer39_out` | `s1_d3_act`  — solver1 final output       | `state2` for track j+1 |

In [9]:
FW = Path(HLS_OUT_DIR) / 'firmware'

# ── Patched myproject.h content ─────────────────────────────────────────────
PATCHED_H = '''\
#ifndef MYPROJECT_H_
#define MYPROJECT_H_

#include "ap_fixed.h"
#include "ap_int.h"
#include "hls_stream.h"

#include "defines.h"

// layer7_out  = emb_out: embed block output   -> feed back as state0 next call
// layer23_out = s0_out:  solver0 final output  -> feed back as state1 next call
// layer39_out = s1_out:  solver1 final output  -> feed back as state2 next call
void myproject(
    hls::stream<input_t>   &track_in,
    hls::stream<input8_t>  &state0,
    hls::stream<input24_t> &state1,
    hls::stream<input40_t> &state2,
    hls::stream<result_t>  &layer55_out,
    hls::stream<layer7_t>  &layer7_out,
    hls::stream<layer23_t> &layer23_out,
    hls::stream<layer39_t> &layer39_out
);

#endif
'''

def apply_firmware_patch(fw_dir):
    """Expose layer7/23/39 as output ports in myproject. Safe to re-run."""
    h_path   = fw_dir / 'myproject.h'
    cpp_path = fw_dir / 'myproject.cpp'
    if not h_path.exists():
        print('Firmware not generated yet — run compile() first.'); return

    # -- patch .h --
    h_path.write_text(PATCHED_H)

    # -- patch .cpp --
    txt = cpp_path.read_text()
    # skip if already patched
    if 'layer7_out,\n    hls::stream<layer23_t>' in txt or 'layer39_out\n)' in txt:
        print('myproject.cpp already patched.'); return
    txt = re.sub(
        r'void myproject\([^)]+\)',
        ('void myproject(\n'
         '    hls::stream<input_t>   &track_in,\n'
         '    hls::stream<input8_t>  &state0,\n'
         '    hls::stream<input24_t> &state1,\n'
         '    hls::stream<input40_t> &state2,\n'
         '    hls::stream<result_t>  &layer55_out,\n'
         '    hls::stream<layer7_t>  &layer7_out,\n'
         '    hls::stream<layer23_t> &layer23_out,\n'
         '    hls::stream<layer39_t> &layer39_out\n)'),
        txt, flags=re.DOTALL
    )
    txt = txt.replace(
        'axis port=track_in,state0,state1,state2,layer55_out',
        'axis port=track_in,state0,state1,state2,layer55_out,layer7_out,layer23_out,layer39_out'
    )
    for decl in [
        'hls::stream<layer7_t> layer7_out("layer7_out");\n    #pragma HLS STREAM variable=layer7_out depth=1\n',
        'hls::stream<layer23_t> layer23_out("layer23_out");\n    #pragma HLS STREAM variable=layer23_out depth=1\n',
        'hls::stream<layer39_t> layer39_out("layer39_out");\n    #pragma HLS STREAM variable=layer39_out depth=1\n',
    ]:
        txt = txt.replace(decl, '')
    cpp_path.write_text(txt)

    # -- patch myproject_test.cpp (Vitis HLS csim testbench) --
    test_path = fw_dir.parent / 'myproject_test.cpp'
    if test_path.exists():
        ttxt = test_path.read_text()
        if 'layer7_out_dummy' not in ttxt:
            ttxt = ttxt.replace(
                'myproject(track_in,state0,state1,state2,layer55_out);',
                'hls::stream<layer7_t>  layer7_out_dummy;\n'
                '    hls::stream<layer23_t> layer23_out_dummy;\n'
                '    hls::stream<layer39_t> layer39_out_dummy;\n'
                '    myproject(track_in,state0,state1,state2,layer55_out,\n'
                '              layer7_out_dummy,layer23_out_dummy,layer39_out_dummy);'
            )
            test_path.write_text(ttxt)
            print(f'myproject_test.cpp patched ({ttxt.count("layer7_out_dummy")} sites)')
        else:
            print('myproject_test.cpp already patched.')


apply_firmware_patch(FW)

# Verify
if FW.exists():
    sig_lines = [l.strip() for l in (FW/'myproject.h').read_text().splitlines()
                 if '&' in l and 'stream' in l]
    print('myproject() ports after patch:')
    for s in sig_lines: print(f'  {s}')


myproject.cpp already patched.
myproject() ports after patch:
  hls::stream<input_t>   &track_in,
  hls::stream<input8_t>  &state0,
  hls::stream<input24_t> &state1,
  hls::stream<input40_t> &state2,
  hls::stream<result_t>  &layer55_out,
  hls::stream<layer7_t>  &layer7_out,
  hls::stream<layer23_t> &layer23_out,
  hls::stream<layer39_t> &layer39_out


In [10]:
# Write rtda_top.cpp to pl/src/
# Run this cell after compile() — it writes the wrapper using the patched myproject() signature.
TOP_PATH = REPO_ROOT / 'pl' / 'src' / 'rtda_top.cpp'

RTDA_TOP = '''\
// rtda_top.cpp
// Processes one event: 50 tracks x 6 features -> 27-dim output.
//
// myproject_stateful() wraps the hls4ml-generated myproject() and owns the
// three state BRAM arrays (emb_prev, s0_prev, s1_prev) as static locals --
// exactly like the AIE roll_concat_kernel uses 'static DATA_TYPE frames[2][N]'.
// No state ever reaches an external port.
//
// rtda_top() external interface:
//   IN  track_stream -- 50 x input_t  (nnet::array<float,6> per track, AXI-S)
//   OUT result_out   -- 1  x result_t  (27 floats in data[0..26], AXI-S)
//   IN  reset        -- pulse high before the first track of each new event

#include \"firmware/myproject.h\"
#include <cstring>

#define N_TRACKS  50
#define HIDDEN    128
#define OUT_DIM   27
#define WARMUP    3

// Output-layer weights (128x27) and bias (27).
// Simulation: loaded from data/*.txt on first call.
// Synthesis:  initialise from ap_uint ROM or expose via AXI-lite slave.
static float OUT_W[HIDDEN][OUT_DIM];
static float OUT_B[OUT_DIM];

#ifndef __SYNTHESIS__
#include <fstream>
#include <stdexcept>
static void _load_output_weights(const char* data_dir) {
    std::ifstream fw(std::string(data_dir) + \"/output_weights.txt\");
    std::ifstream fb(std::string(data_dir) + \"/output_bias.txt\");
    if (!fw || !fb) throw std::runtime_error(\"Cannot open output_weights/bias.txt\");
    for (int k = 0; k < HIDDEN;  k++)
        for (int d = 0; d < OUT_DIM; d++) fw >> OUT_W[k][d];
    for (int d = 0; d < OUT_DIM; d++) fb >> OUT_B[d];
}
static bool _weights_loaded = false;
#endif

// -------------------------------------------------------------------------
// myproject_stateful()
// Stateful shell: owns emb_prev/s0_prev/s1_prev as on-chip BRAM.
// Called once per track. State never leaves this function.
// -------------------------------------------------------------------------
static void myproject_stateful(
    hls::stream<input_t>  &track_in,
    hls::stream<result_t> &s2_out,
    bool reset
) {
    static input8_t  emb_prev;  // emb_out of previous track -> state0
    static input24_t s0_prev;   // s0_out  of previous track -> state1
    static input40_t s1_prev;   // s1_out  of previous track -> state2
#pragma HLS ARRAY_PARTITION variable=emb_prev.data complete
#pragma HLS ARRAY_PARTITION variable=s0_prev.data  complete
#pragma HLS ARRAY_PARTITION variable=s1_prev.data  complete

    if (reset) {
        for (int i = 0; i < HIDDEN; i++) {
#pragma HLS UNROLL
            emb_prev.data[i] = 0.0f;
            s0_prev.data[i]  = 0.0f;
            s1_prev.data[i]  = 0.0f;
        }
    }

    // Pack stored state into streams for myproject()
    hls::stream<input8_t>  st0;  st0.write(emb_prev);
    hls::stream<input24_t> st1;  st1.write(s0_prev);
    hls::stream<input40_t> st2;  st2.write(s1_prev);

    // Capture the three intermediate outputs to update state next call
    hls::stream<layer7_t>  emb_buf;  // emb_out[j]
    hls::stream<layer23_t> s0_buf;   // s0_out[j]
    hls::stream<layer39_t> s1_buf;   // s1_out[j]

    myproject(track_in, st0, st1, st2, s2_out, emb_buf, s0_buf, s1_buf);

    // Update internal BRAM state -- stays on-chip, never leaves
    emb_prev = emb_buf.read();
    s0_prev  = s0_buf.read();
    s1_prev  = s1_buf.read();
}

// -------------------------------------------------------------------------
// rtda_top() -- top-level HLS kernel
// -------------------------------------------------------------------------
void rtda_top(
    hls::stream<input_t>  &track_stream,
    hls::stream<result_t> &result_out,
    bool                   reset,
    const char*            data_dir
) {
#pragma HLS INTERFACE axis       port=track_stream
#pragma HLS INTERFACE axis       port=result_out
#pragma HLS INTERFACE ap_ctrl_hs port=return
#pragma HLS INTERFACE ap_none    port=reset
#pragma HLS INTERFACE ap_none    port=data_dir

#ifndef __SYNTHESIS__
    if (!_weights_loaded) { _load_output_weights(data_dir); _weights_loaded = true; }
#endif

    float acc[HIDDEN];
#pragma HLS ARRAY_PARTITION variable=acc complete
    for (int i = 0; i < HIDDEN; i++) acc[i] = 0.0f;

    for (int j = 0; j < N_TRACKS; j++) {
        hls::stream<result_t> s2_buf;
        // State feedback is entirely inside myproject_stateful().
        // From here it looks like a simple one-in one-out call.
        myproject_stateful(track_stream, s2_buf, reset && (j == 0));
        result_t s2 = s2_buf.read();
        if (j >= WARMUP)
            for (int k = 0; k < HIDDEN; k++)
#pragma HLS UNROLL
                acc[k] += s2.data[k];
    }

    float mean_s2[HIDDEN];
#pragma HLS ARRAY_PARTITION variable=mean_s2 complete
    for (int k = 0; k < HIDDEN; k++)
#pragma HLS UNROLL
        mean_s2[k] = acc[k] / float(N_TRACKS - WARMUP);

    result_t final_out;
    for (int d = 0; d < OUT_DIM; d++) {
        float sum = OUT_B[d];
        for (int k = 0; k < HIDDEN; k++)
            sum += mean_s2[k] * OUT_W[k][d];
        final_out.data[d] = sum;
    }
    for (int d = OUT_DIM; d < HIDDEN; d++) final_out.data[d] = 0.0f;
    result_out.write(final_out);
}
'''

TOP_PATH.write_text(RTDA_TOP)
print(f'Written: {TOP_PATH}')
print()
print('External interface:')
print('  IN  track_stream — 50 x input_t  (nnet::array<float,6> per track)')
print('  OUT result_out   — 1  x result_t  (27 floats in data[0..26])')
print('  IN  reset        — pulse high at start of each new event')
print()
print('State management (fully internal to myproject_stateful):')
print('  static emb_prev[128] <- layer7_out  (emb_out)')
print('  static s0_prev[128]  <- layer23_out (s0_out)')
print('  static s1_prev[128]  <- layer39_out (s1_out)')
print('  No state reaches any external port.')


Written: /home/synthara/VersalPrjs/LDRD/quant_rtda/rtda_demo/pl/src/rtda_top.cpp

External interface:
  IN  track_stream — 50 x input_t  (nnet::array<float,6> per track)
  OUT result_out   — 1  x result_t  (27 floats in data[0..26])
  IN  reset        — pulse high at start of each new event

State management (fully internal to myproject_stateful):
  static emb_prev[128] <- layer7_out  (emb_out)
  static s0_prev[128]  <- layer23_out (s0_out)
  static s1_prev[128]  <- layer39_out (s1_out)
  No state reaches any external port.


## 6. C-Synthesis

Uncomment and run the cell below when you are ready to synthesise.  
Typical wall-clock time: 10–30 min depending on the model size and target clock.

> **Tip**: to synthesise only without re-running csim, use `csim=False, synth=True`.

In [ ]:
# C-Synthesis via vitis-run --mode hls (Vitis 2024.2).
# Uncomment and run when ready — typically 10-30 minutes.
import subprocess

result = subprocess.run(
    ['vitis-run', '--mode', 'hls', '--tcl', 'build_prj.tcl'],
    cwd=HLS_OUT_DIR,
    capture_output=True, text=True
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.stderr: print('STDERR:', result.stderr[-1000:])
print('Return code:', result.returncode)


In [ ]:
hls_model.build(synth=True)

## 7. Synthesis Reports

Run after `hls_model.build(synth=True)` completes.  
Reports are parsed from `<hls_out>/myproject_prj/solution1/syn/report/`.

In [ ]:
# Read synthesis reports — run after C-synthesis completes
try:
    report = hls4ml.report.read_vivado_report(HLS_OUT_DIR)
    print(report)
except Exception as e:
    print(f'No report yet: {e}')

# ── Optional: raw resource table ─────────────────────────────────────────────
import glob
rpt_files = glob.glob(f'{HLS_OUT_DIR}/**/*_csynth.rpt', recursive=True)
for f in rpt_files:
    print(f'\n=== {Path(f).name} ===')
    lines = open(f).readlines()
    in_table = False
    for line in lines:
        if '+ Summary of overall latency' in line or '+ Utilization Estimates' in line:
            in_table = True
        if in_table:
            print(line, end='')
        if in_table and line.strip() == '' and len(line) == 1:
            in_table = False


## 8. Export Keras Model to ONNX

Uses `tf2onnx` to convert the tf_keras model to ONNX opset 13.  
All 4 inputs (`track_in`, `state0/1/2`) and 4 outputs (`s2_out`, `emb_out`, `s0_out`, `s1_out`) are preserved.  
A round-trip check runs each track through the exported ONNX and compares against the saved `keras_s2out.txt` — should be 0.00e+00.

In [ ]:
import tf2onnx
import onnx
import onnxruntime as ort

ONNX_PATH = str(OUT_DIR / 'rtda_per_track.onnx')

# ── Convert ──────────────────────────────────────────────────────────────────
onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature = None,   # inferred from model.inputs
    opset           = 13,
    output_path     = ONNX_PATH,
)

print(f'Exported: {ONNX_PATH}')
print(f'ONNX opset : {onnx_model.opset_import[0].version}')
sess_new = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp_names = [i.name for i in sess_new.get_inputs()]
out_names = [o.name for o in sess_new.get_outputs()]
print(f'Inputs  : {inp_names}')
print(f'Outputs : {out_names}')

# ── Round-trip check vs saved keras_s2out.txt ────────────────────────────────
st = [np.zeros((1, HIDDEN_SIZE), np.float32) for _ in range(3)]
onnx_s2out = np.zeros((N_TRACKS, HIDDEN_SIZE), np.float32)

for j in range(N_TRACKS):
    outs = sess_new.run(None, {
        'track_in': tracks[j:j+1],
        'state0':   st[0], 'state1': st[1], 'state2': st[2],
    })
    s2, eo, o0, o1 = outs
    onnx_s2out[j] = s2.reshape(HIDDEN_SIZE)
    st[0], st[1], st[2] = eo.reshape(1,-1), o0.reshape(1,-1), o1.reshape(1,-1)

diff = np.abs(onnx_s2out - keras_s2out)
print(f'\nONNX vs keras_s2out.txt  max|diff| = {diff.max():.2e}')
print('PASS ✓' if diff.max() < 1e-5 else 'FAIL ✗ — check conversion')
